# Sagittal Landmark Detection — Colab Training

**事前準備:**
1. Colabメニューの「ランタイム → ランタイムのタイプを変更」で **GPU (T4)** を選択
2. `*_image.npy` と `*_landmarks.json` を Google Drive の任意のフォルダにアップロード
3. 下の `DRIVE_DATA_DIR` をそのフォルダパスに変更して全セルを実行

In [ ]:
# ============================================================
# 設定: ここだけ変更すればOK
# ============================================================
DRIVE_DATA_DIR = "/content/drive/MyDrive/anglist_data"  # データフォルダのパス
EPOCHS = 50
BATCH_SIZE = 4
RESIZE = [512, 512]

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# リポジトリのクローンと依存関係インストール
!git clone https://github.com/masaki39/anglist /content/anglist
!pip install -q tqdm onnxruntime onnxscript

In [ ]:
# データをローカルにコピー（Drive経由のI/Oより高速）
import shutil, os
LOCAL_DATA = "/content/data"
if os.path.exists(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA)
files = os.listdir(LOCAL_DATA)
print(f"データファイル数: {len(files)}")
print("\n".join(sorted(files)[:10]))

In [ ]:
# GPU確認
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 訓練実行
!cd /content/anglist/train && python train.py \
    --data-dir {LOCAL_DATA} \
    --save-dir /content/runs \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --resize {RESIZE[0]} {RESIZE[1]}

In [ ]:
# ONNX エクスポート
!cd /content/anglist/train && python export_onnx.py \
    --checkpoint /content/runs/best.pt \
    --output /content/model.onnx \
    --height {RESIZE[0]} \
    --width {RESIZE[1]}

In [ ]:
# モデルをダウンロード
from google.colab import files
files.download('/content/model.onnx')
files.download('/content/runs/best.pt')

In [ ]:
# (オプション) Google Driveにも保存
import shutil
shutil.copy('/content/model.onnx', f"{DRIVE_DATA_DIR}/../model.onnx")
shutil.copy('/content/runs/best.pt', f"{DRIVE_DATA_DIR}/../best.pt")
print("Drive に保存完了")